In [ ]:
# Extracted from SAM, and adopted by Junliang Liu

In [2]:
import torch
import os

if torch.cuda.is_available():
   device = torch.device("cuda")
   print("CUDA is ready for use")

else:
   device = torch.device("cpu")
   print("CPU is ready for use")

print(f"Device selected: {device}\n")

CUDA is ready for use
Device selected: cuda



In [3]:
# import other libraries
import cv2 as cv
import numpy as np
# import matplotlib.pyplot as plt
from sklearn.metrics import accuracy_score
from sklearn.metrics import precision_score
from sklearn.metrics import recall_score
from sklearn.metrics import f1_score

import warnings
warnings.filterwarnings("ignore")

In [ ]:
# read and load the images
# just use validation set to test
val_dir = '/EWS-Dataset/validation'

def load_images(path):
    all_files = os.listdir(path)
    png_files = sorted([f for f in all_files if f.lower().endswith('.png')])
    image_files = [f for f in png_files if not f.lower().endswith('_mask.png')]

    images = []
    masks = []
    
    for fname in image_files:
        img_path = os.path.join(path, fname)
        
        mask_name = fname.replace('.png', '_mask.png')
        mask_path = os.path.join(path, mask_name)

        img = cv.imread(img_path, cv.IMREAD_COLOR)
        mask = cv.imread(mask_path, cv.IMREAD_GRAYSCALE)

        if img is not None and mask is not None:
            img = cv.cvtColor(img, cv.COLOR_BGR2RGB)
            images.append(img)
            masks.append(mask)

    images = np.array(images)
    masks = np.array(masks)

    return images, masks

val_imgs, val_masks = load_images(val_dir)

In [5]:
def compute_metrics(pred_mask, gt_mask):
    pred = (pred_mask > 0).flatten()
    gt   = (gt_mask > 0).flatten()
    
    accuracy  = accuracy_score(gt, pred)
    precision = precision_score(gt, pred)
    recall = recall_score(gt, pred)
    f1 = f1_score(gt, pred)

    # Compute Intersection-over-Union between two binary mask
    intersection = np.logical_and(pred, gt).sum()
    union        = np.logical_or(pred, gt).sum()
    iou = intersection / union if union > 0 else 0.0
    
    return accuracy, precision, recall, f1, iou

In [6]:
from segment_anything import sam_model_registry
from segment_anything import SamAutomaticMaskGenerator

In [ ]:
# load the pre-trained model vit_b
# vit_b is much smaller than vit_h
# saving time
model_type = "vit_b" # base
sam_checkpoint = "sam_vit_b_01ec64.pth"

sam = sam_model_registry[model_type](checkpoint=sam_checkpoint)
sam.to(device=device)

# Model predicts the masks automaticly
mask_generator = SamAutomaticMaskGenerator(
    model=sam,
    points_per_side=16, # 32 -> 16, 
    # saving time, 
    # only focus on block scanning's on/off
    # other parameters are default
    pred_iou_thresh=0.86,
    stability_score_thresh=0.92,
    crop_n_layers=1, # block scanning
    crop_n_points_downscale_factor=2,
    min_mask_region_area=100,
)

# run SAM automatic mask generation and return a binary mask
def predict_sam_mask(image):
    if image.dtype != np.uint8:
        image = image.astype(np.uint8)
    masks = mask_generator.generate(image)

    if len(masks) == 0:
        return np.zeros((image.shape[0], image.shape[1]), dtype=np.uint8)

    # conbine all instance masks
    combined = np.zeros((image.shape[0], image.shape[1]), dtype=bool)
    for m in masks:
        combined = np.logical_or(combined, m['segmentation'])

    return combined.astype(np.uint8) * 255

In [8]:
def evaluation(images, gt_masks, set_name="Validation"):
    pred_masks = []
    for i, img in enumerate(images):
        pred = predict_sam_mask(img)
        pred_masks.append(pred)
    pred_masks = np.array(pred_masks)

    acc_list, prec_list, rec_list, f1_list, iou_list = [], [], [], [], []
    for pred, gt in zip(pred_masks, gt_masks):
        a, p, r, f, i = compute_metrics(pred, gt)
        acc_list.append(a)
        prec_list.append(p)
        rec_list.append(r)
        f1_list.append(f)
        iou_list.append(i)

    print(f"\nSAM (vit_b) Zero-Shot on {set_name}")
    print(f"Accuracy  : {np.mean(acc_list):.4f} ± {np.std(acc_list):.4f}")
    print(f"Precision : {np.mean(prec_list):.4f} ± {np.std(prec_list):.4f}")
    print(f"Recall    : {np.mean(rec_list):.4f} ± {np.std(rec_list):.4f}")
    print(f"F1-score  : {np.mean(f1_list):.4f} ± {np.std(f1_list):.4f}")
    print(f"IoU       : {np.mean(iou_list):.4f} ± {np.std(iou_list):.4f}")
    return pred_masks

In [9]:
# zero-shot evaluation on val, 
# with block scanning
print("SamAutomaticMaskGenerator using block sacnning: ")
val_pred_masks = evaluation(val_imgs, val_masks, "Validation")

SamAutomaticMaskGenerator using block sacnning: 

SAM (vit_b) Zero-Shot on Validation
Accuracy  : 0.3537 ± 0.2192
Precision : 0.5884 ± 0.3105
Recall    : 0.2343 ± 0.2375
F1-score  : 0.2823 ± 0.2151
IoU       : 0.1929 ± 0.2298


In [10]:
del val_pred_masks
torch.cuda.empty_cache()

In [11]:
# re-load the pre-trained model vit_b

model_type = "vit_b" # base
sam_checkpoint = "sam_vit_b_01ec64.pth"

sam = sam_model_registry[model_type](checkpoint=sam_checkpoint)
sam.to(device=device)

# Model predicts the masks automaticly
mask_generator = SamAutomaticMaskGenerator(
    model=sam,
    points_per_side=16,
    pred_iou_thresh=0.86,
    stability_score_thresh=0.92,
    crop_n_layers=0, # global single-pass scanning
    crop_n_points_downscale_factor=2,
    min_mask_region_area=100,
)

# run SAM automatic mask generation and return a binary mask
def predict_sam_mask(image):
    if image.dtype != np.uint8:
        image = image.astype(np.uint8)
    masks = mask_generator.generate(image)

    if len(masks) == 0:
        return np.zeros((image.shape[0], image.shape[1]), dtype=np.uint8)

    # conbine all instance masks
    combined = np.zeros((image.shape[0], image.shape[1]), dtype=bool)
    for m in masks:
        combined = np.logical_or(combined, m['segmentation'])

    return combined.astype(np.uint8) * 255

In [12]:
# zero-shot evaluation on val,
#  with global scanning
print("SamAutomaticMaskGenerator using global single-pass scanning: ")
val_pred_masks = evaluation(val_imgs, val_masks, "Validation")

SamAutomaticMaskGenerator using global single-pass scanning: 

SAM (vit_b) Zero-Shot on Validation
Accuracy  : 0.3495 ± 0.2239
Precision : 0.5670 ± 0.3329
Recall    : 0.1958 ± 0.2449
F1-score  : 0.2444 ± 0.2264
IoU       : 0.1695 ± 0.2367
